In [7]:
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv("../../data/FRB_H15.csv").dropna()

#rename columns
df.rename(columns={"Series Description": "Date", "Market yield on U.S. Treasury securities at 1-month  constant maturity, quoted on investment basis": "0Y1M", "Market yield on U.S. Treasury securities at 3-month  constant maturity, quoted on investment basis": "0Y3M", "Market yield on U.S. Treasury securities at 6-month  constant maturity, quoted on investment basis": "0Y6M", "Market yield on U.S. Treasury securities at 1-year  constant maturity, quoted on investment basis": "1Y", "Market yield on U.S. Treasury securities at 2-year  constant maturity, quoted on investment basis": "2Y", "Market yield on U.S. Treasury securities at 3-year  constant maturity, quoted on investment basis": "3Y", "Market yield on U.S. Treasury securities at 5-year  constant maturity, quoted on investment basis": "5Y", "Market yield on U.S. Treasury securities at 7-year  constant maturity, quoted on investment basis": "7Y", "Market yield on U.S. Treasury securities at 10-year  constant maturity, quoted on investment basis": "10Y", "Market yield on U.S. Treasury securities at 20-year constant maturity, quoted on investment basis": "20Y", "Market yield on U.S. Treasury securities at 30-year  constant maturity, quoted on investment basis": "30Y"}, inplace=True)
df.rename(columns={"Market yield on U.S. Treasury securities at 20-year  constant maturity, quoted on investment basis": "20Y"}, inplace=True)

#make index to datetime for timeseries
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date", inplace=True)

df.tail(20)

,0Y1M,0Y3M,0Y6M,1Y,2Y,3Y,5Y,7Y,10Y,20Y,30Y
Date,,,,,,,,,,,
2026-04-17,3.69,3.70,3.69,3.64,3.71,3.72,3.84,4.04,4.26,4.85,4.88
2026-04-20,3.69,3.71,3.72,3.65,3.72,3.73,3.86,4.04,4.26,4.85,4.88
2026-04-21,3.67,3.69,3.73,3.69,3.78,3.80,3.91,4.09,4.30,4.87,4.89
2026-04-22,3.68,3.69,3.72,3.69,3.79,3.81,3.91,4.10,4.30,4.87,4.90
2026-04-23,3.69,3.69,3.72,3.70,3.83,3.84,3.96,4.13,4.34,4.90,4.92
2026-04-24,3.69,3.69,3.71,3.67,3.78,3.80,3.92,4.10,4.31,4.88,4.91
2026-04-27,3.70,3.68,3.72,3.69,3.78,3.83,3.94,4.14,4.35,4.92,4.94
2026-04-28,3.68,3.68,3.72,3.71,3.84,3.86,3.97,4.16,4.36,4.92,4.94
2026-04-29,3.68,3.68,3.73,3.75,3.92,3.94,4.05,4.23,4.42,4.97,4.98


In [13]:
#split the data into different maturities (for AR model)

trainSplit = {}
#maturities_list = []
for col in df.columns:
    #maturities[col] = df[[col]]
    #maturities_list.append(df[[col]])

    curr = df[[col]]
    trainSplit[col] = {"train": curr[curr.index <= "2025-4-17"], "test": curr[curr.index > "2025-4-17"]} 

    #removes date as indexed column for training data so statsmodels doesnt complain
    trainSplit[col]["train"] = trainSplit[col]["train"].reset_index(drop=True)

#create an AR model for each maturity

modelArr = []

#testing different lag numbers for ideal rmse

matList = ["0Y1M", "0Y3M", "0Y6M", "1Y", "2Y", "3Y", "5Y", "7Y", "10Y", "20Y", "30Y"]
for i in range(len(matList)):
    modelArr.append([])

j=0
for mat in matList:
    
    for i in range(1,31):
        modelArr[j].append( 
            AutoReg(
                endog = trainSplit[mat]["train"][mat],
                lags = i,
                trend = "c"
            )
        )
    j+=1

resultsArr = []
for i in range(len(matList)):
    resultsArr.append([])

i = 0
for arr in modelArr:

    for model in arr:
        resultsArr[i].append(model.fit())
    i+=1


# adds 30 models with lags from 1 - 30 for each maturity to resultsArr
# the index from resultsArr is the index in matList, and index of inner array is lag - 1

In [14]:
#test the model based on y_test

#resultsArr - each arr is a maturity, inner arr is the lags
#trainSplit - key is maturity, from matList, inner dict has train and test dataframes


#forecast maturities so you can compare to actual
forecastArr = []


for i in range(len(resultsArr)):
    forecastArr.append([])
    for result in resultsArr[i]:
        forecastArr[i].append(result.forecast(steps=len(trainSplit[matList[i]]["test"])).to_numpy())

testArr = []
for mat in matList:
    testArr.append(trainSplit[mat]["test"][mat].to_numpy())

#predicted = forecast.to_numpy()

#primary array - each maturity, inner array is error for each given lag
errorsArr = []

i=0
for mat in forecastArr:
    errorsArr.append([])
    for j in range(len(mat)):
        errorsArr[i].append(forecastArr[i][j] - testArr[i])
    i+=1



rmseArr = []
for x in range(len(matList)): 
    rmseArr.append([])
        

for i in range(len(errorsArr)):
    for j in range(len(errorsArr[i])):
        rmseArr[i].append(np.sqrt(np.mean(errorsArr[i][j]**2)))

aggregatedError_bylag = {}
# (dictionary so we can sort the values easily)
for i in range(1,31):
    aggregatedError_bylag[i] = 0
    for j in range(len(matList)):
        aggregatedError_bylag[i] += rmseArr[j][i-1]

for key,value in sorted(aggregatedError_bylag.items(), key = lambda item: item[1]):
    print(key, " : ", value)

29  :  2.4847068556878233
21  :  2.509551268051999
27  :  2.5189575372221706
30  :  2.5266435281494353
28  :  2.5271430732497446
26  :  2.54398081271449
20  :  2.5524444942589133
24  :  2.558715151925793
22  :  2.559250283010447
25  :  2.5675393119200263
16  :  2.569409172945
11  :  2.5876355305037286
17  :  2.592153468324849
23  :  2.5930276461427715
8  :  2.59923304409942
10  :  2.5999957659082265
15  :  2.6027695575921634
7  :  2.6088399632251456
6  :  2.609330874514629
9  :  2.6162217503902014
18  :  2.6195818929055914
2  :  2.622225289001439
12  :  2.632573659696961
14  :  2.638442669253604
3  :  2.6465288041783475
19  :  2.6476198290753477
5  :  2.6650078365307075
13  :  2.66837956210516
1  :  2.7258081566854044
4  :  2.732432537813274
